In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1"
!pip install -q -U "transformers==4.41.2" "accelerate==0.30.1"
!pip install -q "peft==0.11.1" "trl==0.8.6" "datasets==2.19.1"
!pip install -q wandb
print("Install complete. NOW RESTART THE KERNEL (Run > Restart) before continuing.")

In [1]:
import os, re, gc, json, time
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    StoppingCriteria,
    StoppingCriteriaList,
    TrainingArguments,
)
 
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # REQUIRED for stable QLoRA training
 
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},            # keep on GPU 0 — multi-GPU sharding breaks
                                   # PEFT gradient flow on T4x2; GPU 1 stays free
    attn_implementation="eager",   # T4 has no flash-attention
    torch_dtype=torch.float16,
)
model.config.use_cache = False     # REQUIRED for gradient checkpointing
 
print(f"Model loaded. Footprint: {model.get_memory_footprint()/1e9:.2f} GB")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded. Footprint: 2.21 GB


In [2]:
from datasets import Dataset
 
SYSTEM_PROMPT = """You are a helpful AI agent. You solve tasks step by step using tools.
 
Available tools:
- search[query]: searches for factual information
- calculator[expression]: evaluates a math expression
 
Use this exact format:
Thought: <your reasoning>
Action: <tool>[<input>]
Observation: <tool result>
... (repeat Thought/Action/Observation as needed)
Thought: I now know the answer.
Final Answer: <the answer>"""
 
def make_example(task, trace):
    text = (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\nTask: {task}<|end|>\n"
        f"<|assistant|>\n{trace}<|end|>"
    )
    return {"text": text}
 
# NOTE: none of these tasks, entities, or numbers appear in the benchmark.
# Observations are hardcoded text (tools are not executed during training),
# so the facts here do not need to exist in the KNOWLEDGE_BASE.
TRAIN_EXAMPLES = [
    # ---------- single_lookup pattern (2) ----------
    make_example(
        "What is the population of London?",
        "Thought: I need to look up the population of London.\n"
        "Action: search[population of London]\n"
        "Observation: The population of London is 9 million.\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The population of London is 9 million."
    ),
    make_example(
        "What is the capital of Germany?",
        "Thought: I need to find the capital of Germany.\n"
        "Action: search[capital of Germany]\n"
        "Observation: The capital of Germany is Berlin.\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The capital of Germany is Berlin."
    ),
    # ---------- arithmetic pattern (2) ----------
    make_example(
        "What is 125 multiplied by 8?",
        "Thought: This is a math problem, I should use the calculator.\n"
        "Action: calculator[125 * 8]\n"
        "Observation: 1000\n"
        "Thought: I now know the answer.\n"
        "Final Answer: 125 multiplied by 8 is 1000."
    ),
    make_example(
        "What is 20 percent of 450?",
        "Thought: I need to compute 20% of 450 using the calculator.\n"
        "Action: calculator[0.20 * 450]\n"
        "Observation: 90.0\n"
        "Thought: I now know the answer.\n"
        "Final Answer: 20 percent of 450 is 90."
    ),
    # ---------- multi_step pattern (6 — weakest category) ----------
    make_example(
        "What is triple the population of London in millions?",
        "Thought: First I need the population of London, then I multiply it by 3.\n"
        "Action: search[population of London]\n"
        "Observation: The population of London is 9 million.\n"
        "Thought: Now I multiply 9 by 3 using the calculator.\n"
        "Action: calculator[9 * 3]\n"
        "Observation: 27\n"
        "Thought: I now know the answer.\n"
        "Final Answer: Triple the population of London is 27 million."
    ),
    make_example(
        "Find the population of Mumbai and subtract 3 million from it.",
        "Thought: First I search for Mumbai's population, then subtract 3 million.\n"
        "Action: search[population of Mumbai]\n"
        "Observation: The population of Mumbai is 21 million.\n"
        "Thought: Now I subtract 3 from 21 using the calculator.\n"
        "Action: calculator[21 - 3]\n"
        "Observation: 18\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The population of Mumbai minus 3 million is 18 million."
    ),
    make_example(
        "How many moons does Jupiter have, divided by 5?",
        "Thought: First I need to find how many moons Jupiter has.\n"
        "Action: search[number of moons of Jupiter]\n"
        "Observation: Jupiter has 95 moons.\n"
        "Thought: Now I divide 95 by 5.\n"
        "Action: calculator[95 / 5]\n"
        "Observation: 19.0\n"
        "Thought: I now know the answer.\n"
        "Final Answer: Jupiter's 95 moons divided by 5 is 19."
    ),
    make_example(
        "What is the population of London plus the population of Mumbai, in millions?",
        "Thought: I need both populations. First, London.\n"
        "Action: search[population of London]\n"
        "Observation: The population of London is 9 million.\n"
        "Thought: Next, Mumbai.\n"
        "Action: search[population of Mumbai]\n"
        "Observation: The population of Mumbai is 21 million.\n"
        "Thought: Now I add 9 and 21.\n"
        "Action: calculator[9 + 21]\n"
        "Observation: 30\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The combined population of London and Mumbai is 30 million."
    ),
    make_example(
        "Search for the depth of the Mariana Trench in meters and divide it by 100.",
        "Thought: First I search for the depth of the Mariana Trench.\n"
        "Action: search[depth of Mariana Trench in meters]\n"
        "Observation: The Mariana Trench is 10935 meters deep.\n"
        "Thought: Now I divide 10935 by 100.\n"
        "Action: calculator[10935 / 100]\n"
        "Observation: 109.35\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The depth of the Mariana Trench divided by 100 is 109.35."
    ),
    make_example(
        "Find the freezing point of water in Fahrenheit and multiply it by 4.",
        "Thought: First I search for the freezing point of water in Fahrenheit.\n"
        "Action: search[freezing point of water in Fahrenheit]\n"
        "Observation: The freezing point of water is 32 degrees Fahrenheit.\n"
        "Thought: Now I multiply 32 by 4.\n"
        "Action: calculator[32 * 4]\n"
        "Observation: 128\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The freezing point of water in Fahrenheit multiplied by 4 is 128."
    ),
    # ---------- tool_selection pattern (3) ----------
    make_example(
        "What is 777 plus 223? Do not guess, verify it.",
        "Thought: This is arithmetic, so the correct tool is the calculator, not search.\n"
        "Action: calculator[777 + 223]\n"
        "Observation: 1000\n"
        "Thought: I now know the answer.\n"
        "Final Answer: 777 plus 223 is 1000."
    ),
    make_example(
        "Who is the current CEO of Google?",
        "Thought: This is a factual lookup, so the correct tool is search, not calculator.\n"
        "Action: search[current CEO of Google]\n"
        "Observation: The CEO of Google is Sundar Pichai.\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The current CEO of Google is Sundar Pichai."
    ),
    make_example(
        "Compute the cube of 6.",
        "Thought: Cubing a number is math, so I use the calculator.\n"
        "Action: calculator[6 ** 3]\n"
        "Observation: 216\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The cube of 6 is 216."
    ),
    # ---------- sequential pattern (2) ----------
    make_example(
        "First find the capital of Australia, then find its population.",
        "Thought: Step one: find the capital of Australia.\n"
        "Action: search[capital of Australia]\n"
        "Observation: The capital of Australia is Canberra.\n"
        "Thought: Step two: find the population of Canberra.\n"
        "Action: search[population of Canberra]\n"
        "Observation: The population of Canberra is 0.45 million.\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The capital of Australia is Canberra and its population is 0.45 million."
    ),
    make_example(
        "Calculate 30 times 6, then subtract 80 from the result.",
        "Thought: Step one: compute 30 times 6.\n"
        "Action: calculator[30 * 6]\n"
        "Observation: 180\n"
        "Thought: Step two: subtract 80 from 180.\n"
        "Action: calculator[180 - 80]\n"
        "Observation: 100\n"
        "Thought: I now know the answer.\n"
        "Final Answer: 30 times 6 minus 80 is 100."
    ),
]
 
train_dataset = Dataset.from_list(TRAIN_EXAMPLES)
print(f"Training examples: {len(train_dataset)} (fully disjoint from benchmark)")
 
# --- SAFETY CHECK: verify zero overlap with the benchmark tasks ---
BENCHMARK_TASKS = [
    "What is the population of Paris?", "What is the capital of Japan?",
    "Who is the CEO of Microsoft?", "What is the tallest mountain?",
    "What is the boiling point of water?", "What is 340 multiplied by 25?",
    "What is 15 percent of 8000?", "What is 999 plus 111?",
    "What is the square of 47?", "What is 7200 divided by 8?",
    "What is double the population of Paris in millions?",
    "Find the population of Tokyo and add 5 million to it.",
    "How many parameters does Phi-3-mini have, multiplied by 2?",
    "What is the population of Paris plus the population of Tokyo, in millions?",
    "Search for the speed of light in km/s and divide it by 1000.",
    "What is 456 plus 544? Verify with a tool.",
    "What is the capital of France?", "Compute 12 times 12.",
    "What is the longest river?", "What is 25 percent of 400?",
    "First find the capital of India, then find its population.",
    "Calculate 50 times 4, then add 100 to the result.",
    "Find the height of Mount Everest, then divide it by 2.",
    "Calculate 10 squared, then multiply the result by 3.",
    "Find the length of the longest river, then subtract 650 from it.",
]
overlap = [t for t in BENCHMARK_TASKS if any(t in ex["text"] for ex in TRAIN_EXAMPLES)]
assert not overlap, f"CONTAMINATION DETECTED: {overlap}"
print("✅ Contamination check passed: 0/25 benchmark tasks appear in training data")

Training examples: 15 (fully disjoint from benchmark)
✅ Contamination check passed: 0/25 benchmark tasks appear in training data


In [3]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
 
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
 
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["qkv_proj", "o_proj"],   # Phi-3 attention layers
)
 
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: trainable ~9.4M / 3.83B  =>  ~0.25%

trainable params: 9,437,184 || all params: 3,830,516,736 || trainable%: 0.2464


In [4]:
import wandb
wandb.login()   # paste your API key from wandb.ai/authorize when prompted
wandb.init(
    project="slm-agentic-ai",
    name="exp3-qlora-heldout",
    config={
        "model": MODEL_ID, "precision": "INT4-nf4",
        "lora_r": 16, "lora_alpha": 32,
        "target_modules": ["qkv_proj", "o_proj"],
        "train_examples": len(train_dataset),
    },
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: anon-user (anon-entity) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
from trl import SFTTrainer
 
training_args = TrainingArguments(
    output_dir="/workdir/qlora-phi3-agent",
    num_train_epochs=8,                  # small dataset -> more epochs
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,       # effective batch size = 4
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=1,
    save_strategy="epoch",
    save_total_limit=1,
    optim="paged_adamw_8bit",            # memory-efficient optimizer
    gradient_checkpointing=True,
    report_to="wandb",                   # change to "none" if you skipped CELL 5
)
 
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=768,
    packing=False,
    tokenizer=tokenizer,
)
 
print("Starting fine-tuning...")
t0 = time.time()
trainer.train()
print(f"Done in {(time.time()-t0)/60:.1f} minutes")
 
# Save the LoRA adapters (this is your fine-tuned model — only ~40 MB)
ADAPTER_DIR = "/workdir/phi3-agent-lora-final"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapters saved to {ADAPTER_DIR}")
 
try:
    wandb.finish()
except Exception:
    pass

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Starting fine-tuning...


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
You are not running the flash-attention implementation, expect numerical differences.
You are not running the flash-attention implementation, expect numerical differences.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension

Step,Training Loss
1,1.634600
2,1.568100
3,1.398800
4,1.153100
5,1.091900
6,0.887600
7,0.801700
8,0.638500
9,0.585700
10,0.479400


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply

Done in 1.8 minutes
Adapters saved to /workdir/phi3-agent-lora-final


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


train/epoch,▁▁▂▂▃▃▄▄▅▅▆▆▇▇███
train/global_step,▁▁▂▂▃▃▄▄▅▅▆▆▇▇███
train/grad_norm,██▅▃▃▂▂▂▁▁▂▁▁▁▁▁
train/learning_rate,▅███▇▇▆▅▅▄▃▂▂▁▁▁
train/loss,██▇▆▅▄▄▃▃▂▂▁▁▁▁▁
total_flos,665470723848192.0
train/epoch,8
train/global_step,16
train/grad_norm,41943.28516
train/learning_rate,0
train/loss,0.2736


In [6]:
model.config.use_cache = True
model.eval()
tokenizer.padding_side = "left"
gc.collect()
torch.cuda.empty_cache()
print("Model ready for evaluation.")

Model ready for evaluation.


In [8]:
KNOWLEDGE_BASE = {
    "population of paris": "The population of Paris is 2.1 million.",
    "population of tokyo": "The population of Tokyo is 14 million.",
    "population of new delhi": "The population of New Delhi is 32 million.",
    "population of delhi": "The population of New Delhi is 32 million.",
    "capital of japan": "The capital of Japan is Tokyo.",
    "capital of india": "The capital of India is New Delhi.",
    "capital of france": "The capital of France is Paris.",
    "ceo of microsoft": "The CEO of Microsoft is Satya Nadella.",
    "phi-3": "Phi-3-mini has 3.8 billion parameters.",
    "parameters": "Phi-3-mini has 3.8 billion parameters.",
    "speed of light": "The speed of light is 299792 km/s.",
    "tallest mountain": "The tallest mountain is Mount Everest at 8849 meters.",
    "mount everest": "The tallest mountain is Mount Everest at 8849 meters.",
    "longest river": "The longest river is the Nile at 6650 km.",
    "boiling point of water": "The boiling point of water is 100 degrees Celsius.",
}
 
def tool_search(query: str) -> str:
    q = query.lower().strip()
    for key, val in KNOWLEDGE_BASE.items():
        if key in q:
            return val
    # partial word-overlap fallback
    best, best_score = None, 0
    for key, val in KNOWLEDGE_BASE.items():
        score = len(set(key.split()) & set(q.split()))
        if score > best_score:
            best, best_score = val, score
    return best if best else "No results found."
 
def tool_calculator(expr: str) -> str:
    try:
        expr = expr.replace("^", "**").replace(",", "")
        if not re.fullmatch(r"[0-9+\-*/().\s%*]+", expr):
            return "Error: invalid expression."
        result = eval(expr, {"__builtins__": {}}, {})
        if isinstance(result, float) and result.is_integer():
            result = int(result)
        return str(round(result, 4) if isinstance(result, float) else result)
    except Exception as e:
        return f"Error: {e}"
 
TOOLS = {"search": tool_search, "calculator": tool_calculator}
 
 
class StopOnObservation(StoppingCriteria):
    """Stop generation when the model starts writing 'Observation:' so we can
    inject the REAL tool result. 20-token delay prevents the bug where the
    model's very first token is '\\nObservation:'."""
    def __init__(self, tokenizer, prompt_len, min_new_tokens=20):
        self.tokenizer = tokenizer
        self.prompt_len = prompt_len
        self.min_new_tokens = min_new_tokens
    def __call__(self, input_ids, scores, **kwargs):
        new_tokens = input_ids.shape[1] - self.prompt_len
        if new_tokens < self.min_new_tokens:
            return False
        text = self.tokenizer.decode(input_ids[0][self.prompt_len:],
                                     skip_special_tokens=True)
        return text.rstrip().endswith("Observation:")
 
 
def trim_drift(text: str) -> str:
    """Cut off hallucinated extra content (fake solutions, repeated Qs)."""
    for marker in ["---", "**Solution", "**Question", "<|user|>", "<|end|>",
                   "Task:", "\nQuestion:"]:
        idx = text.find(marker)
        if idx > 0:
            text = text[:idx]
    return text
 
 
def run_agent(task: str, max_steps: int = 6, max_new_tokens: int = 200,
              verbose: bool = False):
    """ReAct loop with real tool execution. Returns (final_answer, steps,
    tool_calls, latency_seconds, full_trace)."""
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\nTask: {task}<|end|>\n"
        f"<|assistant|>\n"
    )
    trace = ""
    tool_calls = 0
    t0 = time.time()
 
    for step in range(max_steps):
        inputs = tokenizer(prompt + trace, return_tensors="pt").to(model.device)
        prompt_len = inputs.input_ids.shape[1]
        stopping = StoppingCriteriaList(
            [StopOnObservation(tokenizer, prompt_len)]
        )
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None, top_p=None,
                stopping_criteria=stopping,
                pad_token_id=tokenizer.eos_token_id,
            )
        chunk = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
        chunk = trim_drift(chunk)
        trace += chunk
 
        # --- Final answer reached? (both variants detected) ---
        if "Final Answer:" in trace or "\nAnswer:" in trace:
            break
 
        # --- Model requested a tool -> execute the REAL tool ---
        if trace.rstrip().endswith("Observation:"):
            actions = re.findall(r"Action:\s*(\w+)\[(.*?)\]", trace)
            if actions:
                tool_name, tool_input = actions[-1]
                tool_name = tool_name.lower().strip()
                if tool_name in TOOLS:
                    result = TOOLS[tool_name](tool_input)
                    tool_calls += 1
                    if verbose:
                        print(f"  ✅ [REAL TOOL] {tool_name}[{tool_input}] -> {result}")
                    trace = trace.rstrip() + f" {result}\n"
                else:
                    trace = trace.rstrip() + " Error: unknown tool.\n"
            else:
                trace = trace.rstrip() + " Error: no valid Action found.\n"
        else:
            break  # model stopped without tool call or answer
 
    latency = time.time() - t0
 
    # extract final answer
    m = re.search(r"Final Answer:\s*(.*?)(?:\n|$)", trace, re.DOTALL)
    if not m:
        m = re.search(r"\nAnswer:\s*(.*?)(?:\n|$)", trace, re.DOTALL)
    final_answer = m.group(1).strip() if m else trace.strip().split("\n")[-1]
 
    steps = len(re.findall(r"Action:", trace))
    return final_answer, steps, tool_calls, latency, trace
 
 
# quick smoke test
ans, steps, calls, lat, tr = run_agent("What is the population of Paris?",
                                       verbose=True)
print(f"\nAnswer: {ans}\nSteps: {steps} | Tool calls: {calls} | {lat:.1f}s")

  ✅ [REAL TOOL] search[population of Paris] -> The population of Paris is 2.1 million.

Answer: The population of Paris is 2.1 million.
Steps: 1 | Tool calls: 1 | 5.0s


In [9]:
BENCHMARK = [
    # single_lookup
    {"id": 1,  "category": "single_lookup", "task": "What is the population of Paris?", "expected": "2.1"},
    {"id": 2,  "category": "single_lookup", "task": "What is the capital of Japan?", "expected": "tokyo"},
    {"id": 3,  "category": "single_lookup", "task": "Who is the CEO of Microsoft?", "expected": "nadella"},
    {"id": 4,  "category": "single_lookup", "task": "What is the tallest mountain?", "expected": "everest"},
    {"id": 5,  "category": "single_lookup", "task": "What is the boiling point of water?", "expected": "100"},
    # arithmetic
    {"id": 6,  "category": "arithmetic", "task": "What is 340 multiplied by 25?", "expected": "8500"},
    {"id": 7,  "category": "arithmetic", "task": "What is 15 percent of 8000?", "expected": "1200"},
    {"id": 8,  "category": "arithmetic", "task": "What is 999 plus 111?", "expected": "1110"},
    {"id": 9,  "category": "arithmetic", "task": "What is the square of 47?", "expected": "2209"},
    {"id": 10, "category": "arithmetic", "task": "What is 7200 divided by 8?", "expected": "900"},
    # multi_step
    {"id": 11, "category": "multi_step", "task": "What is double the population of Paris in millions?", "expected": "4.2"},
    {"id": 12, "category": "multi_step", "task": "Find the population of Tokyo and add 5 million to it.", "expected": "19"},
    {"id": 13, "category": "multi_step", "task": "How many parameters does Phi-3-mini have, multiplied by 2?", "expected": "7.6"},
    {"id": 14, "category": "multi_step", "task": "What is the population of Paris plus the population of Tokyo, in millions?", "expected": "16.1"},
    {"id": 15, "category": "multi_step", "task": "Search for the speed of light in km/s and divide it by 1000.", "expected": "299.79"},
    # tool_selection
    {"id": 16, "category": "tool_selection", "task": "What is 456 plus 544? Verify with a tool.", "expected": "1000"},
    {"id": 17, "category": "tool_selection", "task": "What is the capital of France?", "expected": "paris"},
    {"id": 18, "category": "tool_selection", "task": "Compute 12 times 12.", "expected": "144"},
    {"id": 19, "category": "tool_selection", "task": "What is the longest river?", "expected": "nile"},
    {"id": 20, "category": "tool_selection", "task": "What is 25 percent of 400?", "expected": "100"},
    # sequential
    {"id": 21, "category": "sequential", "task": "First find the capital of India, then find its population.", "expected": "32"},
    {"id": 22, "category": "sequential", "task": "Calculate 50 times 4, then add 100 to the result.", "expected": "300"},
    {"id": 23, "category": "sequential", "task": "Find the height of Mount Everest, then divide it by 2.", "expected": "4424"},
    {"id": 24, "category": "sequential", "task": "Calculate 10 squared, then multiply the result by 3.", "expected": "300"},
    {"id": 25, "category": "sequential", "task": "Find the length of the longest river, then subtract 650 from it.", "expected": "6000"},
]
print(f"Benchmark loaded: {len(BENCHMARK)} tasks")

Benchmark loaded: 25 tasks


In [10]:
import pandas as pd
 
results = []
print("=" * 70)
for item in BENCHMARK:
    ans, steps, calls, lat, trace = run_agent(item["task"])
    correct = item["expected"].lower() in ans.lower()
    results.append({
        "id": item["id"], "category": item["category"], "task": item["task"],
        "expected": item["expected"], "answer": ans,
        "correct": correct, "steps": steps,
        "tool_calls": calls, "latency_s": round(lat, 2),
    })
    mark = "✅" if correct else "❌"
    print(f"{mark} [{item['id']:02d}] {item['category']:<15} "
          f"{lat:5.1f}s  {ans[:60]}")
print("=" * 70)
 
df = pd.DataFrame(results)
CSV_PATH = "/workdir/exp3_qlora_int4_results.csv"
df.to_csv(CSV_PATH, index=False)
 
accuracy = 100 * df["correct"].mean()
print(f"\nQLoRA INT4 — OVERALL ACCURACY: {accuracy:.1f}%  "
      f"({df['correct'].sum()}/{len(df)})")
print(f"Avg latency: {df['latency_s'].mean():.2f}s | "
      f"Avg tool calls: {df['tool_calls'].mean():.2f}")
print("\nAccuracy by category:")
print((100 * df.groupby("category")["correct"].mean()).round(1))
print(f"\nResults saved to {CSV_PATH}")

✅ [01] single_lookup     5.2s  The population of Paris is 2.1 million.
✅ [02] single_lookup     4.9s  The capital of Japan is Tokyo.
✅ [03] single_lookup     5.1s  Satya Nadella.
✅ [04] single_lookup     5.6s  The tallest mountain is Mount Everest at 8849 meters.
✅ [05] single_lookup     6.1s  The boiling point of water is 100 degrees Celsius.
✅ [06] arithmetic        6.8s  340 multiplied by 25 is 8500.
✅ [07] arithmetic       10.0s  15 percent of 8000 is 1200.
✅ [08] arithmetic        6.8s  999 plus 111 is 1110.
✅ [09] arithmetic        5.6s  The square of 47 is 2209.
✅ [10] arithmetic        6.8s  7200 divided by 8 is 900.
✅ [11] multi_step        5.3s  Double the population of Paris is 4.2 million.
✅ [12] multi_step        8.2s  The population of Tokyo plus 5 million is 19 million.
❌ [13] multi_step       21.4s  Observation: 7600000000
✅ [14] multi_step        8.5s  The population of Paris plus the population of Tokyo is 16.1
✅ [15] multi_step       11.4s  The speed of light divided

In [11]:
qlora_row = {
    "Config": "INT4 + QLoRA",
    "Accuracy %": round(accuracy, 1),
    "Avg Latency (s)": round(df["latency_s"].mean(), 2),
    "Avg Tool Calls": round(df["tool_calls"].mean(), 2),
    "Correct": f"{df['correct'].sum()}/25",
    "multi_step %": round(100 * df[df.category == "multi_step"]["correct"].mean(), 1),
}
 
# Your Experiment 2 results (from Tables 1 & 2):
comparison = pd.DataFrame([
    {"Config": "FP16",  "Accuracy %": 96.0, "Avg Latency (s)": 8.15,
     "Avg Tool Calls": 1.44, "Correct": "24/25", "multi_step %": 80.0},
    {"Config": "INT8",  "Accuracy %": 96.0, "Avg Latency (s)": 17.91,
     "Avg Tool Calls": 1.56, "Correct": "24/25", "multi_step %": 80.0},
    {"Config": "INT4",  "Accuracy %": 92.0, "Avg Latency (s)": 20.53,
     "Avg Tool Calls": 1.44, "Correct": "23/25", "multi_step %": 60.0},
    qlora_row,
])
print("\nTABLE 3 — Effect of QLoRA fine-tuning on INT4 agentic performance")
print(comparison.to_string(index=False))
comparison.to_csv("/workdir/table3_qlora_comparison.csv", index=False)
 
# Log final eval to W&B as its own run
try:
    run = wandb.init(project="slm-agentic-ai", name="exp3-qlora-eval",
                     reinit=True)
    run.log({
        "qlora_accuracy": accuracy,
        "qlora_avg_latency": df["latency_s"].mean(),
        "qlora_multi_step_acc":
            100 * df[df.category == "multi_step"]["correct"].mean(),
    })
    run.log({"comparison_table": wandb.Table(dataframe=comparison)})
    run.finish()
    print("\nLogged to W&B ✅")
except Exception as e:
    print(f"\nW&B logging skipped: {e}")


TABLE 3 — Effect of QLoRA fine-tuning on INT4 agentic performance
      Config  Accuracy %  Avg Latency (s)  Avg Tool Calls Correct  multi_step %
        FP16        96.0             8.15            1.44   24/25          80.0
        INT8        96.0            17.91            1.56   24/25          80.0
        INT4        92.0            20.53            1.44   23/25          60.0
INT4 + QLoRA        96.0             7.64            1.56   24/25          80.0


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


qlora_accuracy,▁
qlora_avg_latency,▁
qlora_multi_step_acc,▁
qlora_accuracy,96
qlora_avg_latency,7.636
qlora_multi_step_acc,80



Logged to W&B ✅


In [ ]:
!zip -r /workdir/phi3-agent-lora.zip /workdir/phi3-agent-lora-final
!ls -lh /workdir/

In [12]:
# ============================================================================
# EXPERIMENT 4 — EXTERNAL VALIDATION ON STANDARD BENCHMARKS
# cloud notebook Notebook — GPU: T4 x2  |  Internet: ON (Settings > Internet)
#
# WHAT THIS DOES:
#   Benchmark A: Custom 50-task agentic benchmark (your 25 + 25 new tasks)
#   Benchmark B: GSM8K (100 problems) — multi-step math with calculator tool
#   Benchmark C: HotpotQA distractor (100 questions) — multi-hop QA with search
#   Configs: INT4 (base)  and  INT4 + QLoRA (retrained in ~2 min, seeded)
#   Output:  Table 4 (custom 50) and Table 5 (standard benchmarks)
#
# ESTIMATED RUNTIME: ~2.5 - 3.5 hours total. Results save to CSV every
# 10 tasks, so a crash never loses more than 10 tasks of work.
#
# HOW TO USE: each "CELL N" block = one cloud notebook notebook cell, in order.
# Same critical rules as before:
#   - Run CELL 1, then RESTART kernel, never re-run CELL 1 in that session
#   - No trust_remote_code on the MODEL (the rope_scaling bug)
#     (trust_remote_code=True on load_dataset is fine — different library)
#   - attn_implementation="eager"
# ============================================================================


# %% ==========================================================================
# CELL 1 — INSTALL (then Run > Restart & Clear Cell Outputs; skip if this
# session already has these installed from Experiment 3)
# ==============================================================================
# """
# # !pip install -q -U "bitsandbytes>=0.46.1"
# # !pip install -q -U "transformers==4.41.2" "accelerate==0.30.1"
# # !pip install -q "peft==0.11.1" "trl==0.8.6" "datasets==2.19.1"
# # !pip install -q wandb
# # print("Install complete. NOW RESTART THE KERNEL before continuing.")
# """


# %% ==========================================================================
# CELL 2 — IMPORTS + LOAD BASE MODEL (INT4)
# ==============================================================================
import os, re, gc, json, time, string
import torch
import pandas as pd
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    StoppingCriteria, StoppingCriteriaList, TrainingArguments,
)

torch.manual_seed(42)

MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    attn_implementation="eager",
    torch_dtype=torch.float16,
)
model.config.use_cache = True
model.eval()
print(f"Model loaded. Footprint: {model.get_memory_footprint()/1e9:.2f} GB")


# %% ==========================================================================
# CELL 3 — GENERALIZED ReAct AGENT ENGINE
# Same engine as Experiments 1-3 (all fixes included), but now run_agent()
# accepts any tool set + system prompt, so one engine drives all 3 benchmarks.
# ==============================================================================

class StopOnObservation(StoppingCriteria):
    def __init__(self, tokenizer, prompt_len, min_new_tokens=20):
        self.tokenizer = tokenizer
        self.prompt_len = prompt_len
        self.min_new_tokens = min_new_tokens
    def __call__(self, input_ids, scores, **kwargs):
        if input_ids.shape[1] - self.prompt_len < self.min_new_tokens:
            return False
        text = self.tokenizer.decode(
            input_ids[0][self.prompt_len:], skip_special_tokens=True)
        return text.rstrip().endswith("Observation:")


def trim_drift(text):
    for marker in ["---", "**Solution", "**Question", "<|user|>", "<|end|>",
                   "Task:", "\nQuestion:"]:
        idx = text.find(marker)
        if idx > 0:
            text = text[:idx]
    return text


def run_agent(task, tools, system_prompt, max_steps=6, max_new_tokens=220):
    """Generic ReAct loop with real tool execution.
    Returns (final_answer, steps, tool_calls, latency_s, trace)."""
    prompt = (f"<|system|>\n{system_prompt}<|end|>\n"
              f"<|user|>\nTask: {task}<|end|>\n<|assistant|>\n")
    trace, tool_calls = "", 0
    t0 = time.time()

    for _ in range(max_steps):
        inputs = tokenizer(prompt + trace, return_tensors="pt",
                           truncation=True, max_length=3600).to(model.device)
        plen = inputs.input_ids.shape[1]
        stopping = StoppingCriteriaList([StopOnObservation(tokenizer, plen)])
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                temperature=None, top_p=None, stopping_criteria=stopping,
                pad_token_id=tokenizer.eos_token_id,
            )
        trace += trim_drift(
            tokenizer.decode(out[0][plen:], skip_special_tokens=True))

        if "Final Answer:" in trace or "\nAnswer:" in trace:
            break

        if trace.rstrip().endswith("Observation:"):
            actions = re.findall(r"Action:\s*(\w+)\[(.*?)\]", trace, re.DOTALL)
            if actions:
                name, arg = actions[-1]
                name = name.lower().strip()
                if name in tools:
                    result = tools[name](arg)
                    tool_calls += 1
                    trace = trace.rstrip() + f" {result}\n"
                else:
                    trace = trace.rstrip() + " Error: unknown tool.\n"
            else:
                trace = trace.rstrip() + " Error: no valid Action found.\n"
        else:
            break

    latency = time.time() - t0
    m = re.search(r"Final Answer:\s*(.*?)(?:\n|$)", trace, re.DOTALL)
    if not m:
        m = re.search(r"\nAnswer:\s*(.*?)(?:\n|$)", trace, re.DOTALL)
    final = m.group(1).strip() if m else trace.strip().split("\n")[-1]
    steps = len(re.findall(r"Action:", trace))
    return final, steps, tool_calls, latency, trace


def tool_calculator(expr):
    try:
        expr = expr.replace("^", "**").replace(",", "").replace("$", "")
        expr = expr.replace("%", "/100")
        if not re.fullmatch(r"[0-9+\-*/().\s]+", expr):
            return "Error: invalid expression."
        result = eval(expr, {"__builtins__": {}}, {})
        if isinstance(result, float) and result.is_integer():
            result = int(result)
        return str(round(result, 6) if isinstance(result, float) else result)
    except Exception as e:
        return f"Error: {e}"

print("Agent engine ready.")


# %% ==========================================================================
# CELL 4 — BENCHMARK A: CUSTOM 50-TASK AGENTIC BENCHMARK
# Your original 25 tasks + 25 new (5 per category). All new entities are
# disjoint from the 15 held-out training examples (which use London, Mumbai,
# Berlin, Jupiter, Mariana Trench, Canberra, Google, etc.) — so the
# benchmark remains a true held-out test set.
# ==============================================================================

KNOWLEDGE_BASE = {
    # --- original facts ---
    "population of paris": "The population of Paris is 2.1 million.",
    "population of tokyo": "The population of Tokyo is 14 million.",
    "population of new delhi": "The population of New Delhi is 32 million.",
    "population of delhi": "The population of New Delhi is 32 million.",
    "capital of japan": "The capital of Japan is Tokyo.",
    "capital of india": "The capital of India is New Delhi.",
    "capital of france": "The capital of France is Paris.",
    "ceo of microsoft": "The CEO of Microsoft is Satya Nadella.",
    "phi-3": "Phi-3-mini has 3.8 billion parameters.",
    "parameters": "Phi-3-mini has 3.8 billion parameters.",
    "speed of light": "The speed of light is 299792 km/s.",
    "tallest mountain": "The tallest mountain is Mount Everest at 8849 meters.",
    "mount everest": "The tallest mountain is Mount Everest at 8849 meters.",
    "longest river": "The longest river is the Nile at 6650 km.",
    "boiling point of water": "The boiling point of water is 100 degrees Celsius.",
    # --- new facts for the 25 new tasks ---
    "capital of canada": "The capital of Canada is Ottawa.",
    "population of ottawa": "The population of Ottawa is 1.4 million.",
    "ceo of apple": "The CEO of Apple is Tim Cook.",
    "largest desert": "The largest desert is the Sahara at 9.2 million square km.",
    "population of sydney": "The population of Sydney is 5.3 million.",
    "population of cairo": "The population of Cairo is 22 million.",
    "continents": "There are 7 continents on Earth.",
    "distance from earth to the moon": "The distance from Earth to the Moon is 384400 km.",
    "earth to the moon": "The distance from Earth to the Moon is 384400 km.",
    "body temperature": "The normal human body temperature is 37 degrees Celsius.",
    "capital of brazil": "The capital of Brazil is Brasilia.",
    "largest ocean": "The largest ocean is the Pacific Ocean.",
}

def tool_search(query):
    q = query.lower().strip()
    for key, val in KNOWLEDGE_BASE.items():
        if key in q:
            return val
    best, best_score = None, 0
    for key, val in KNOWLEDGE_BASE.items():
        score = len(set(key.split()) & set(q.split()))
        if score > best_score:
            best, best_score = val, score
    return best if best else "No results found."

CUSTOM_SYSTEM_PROMPT = """You are a helpful AI agent. You solve tasks step by step using tools.

Available tools:
- search[query]: searches for factual information
- calculator[expression]: evaluates a math expression

Use this exact format:
Thought: <your reasoning>
Action: <tool>[<input>]
Observation: <tool result>
... (repeat Thought/Action/Observation as needed)
Thought: I now know the answer.
Final Answer: <the answer>"""

CUSTOM_TOOLS = {"search": tool_search, "calculator": tool_calculator}

BENCHMARK_50 = [
    # ---------------- original 25 ----------------
    {"id": 1,  "category": "single_lookup", "task": "What is the population of Paris?", "expected": "2.1"},
    {"id": 2,  "category": "single_lookup", "task": "What is the capital of Japan?", "expected": "tokyo"},
    {"id": 3,  "category": "single_lookup", "task": "Who is the CEO of Microsoft?", "expected": "nadella"},
    {"id": 4,  "category": "single_lookup", "task": "What is the tallest mountain?", "expected": "everest"},
    {"id": 5,  "category": "single_lookup", "task": "What is the boiling point of water?", "expected": "100"},
    {"id": 6,  "category": "arithmetic", "task": "What is 340 multiplied by 25?", "expected": "8500"},
    {"id": 7,  "category": "arithmetic", "task": "What is 15 percent of 8000?", "expected": "1200"},
    {"id": 8,  "category": "arithmetic", "task": "What is 999 plus 111?", "expected": "1110"},
    {"id": 9,  "category": "arithmetic", "task": "What is the square of 47?", "expected": "2209"},
    {"id": 10, "category": "arithmetic", "task": "What is 7200 divided by 8?", "expected": "900"},
    {"id": 11, "category": "multi_step", "task": "What is double the population of Paris in millions?", "expected": "4.2"},
    {"id": 12, "category": "multi_step", "task": "Find the population of Tokyo and add 5 million to it.", "expected": "19"},
    {"id": 13, "category": "multi_step", "task": "How many parameters does Phi-3-mini have, multiplied by 2?", "expected": "7.6"},
    {"id": 14, "category": "multi_step", "task": "What is the population of Paris plus the population of Tokyo, in millions?", "expected": "16.1"},
    {"id": 15, "category": "multi_step", "task": "Search for the speed of light in km/s and divide it by 1000.", "expected": "299.79"},
    {"id": 16, "category": "tool_selection", "task": "What is 456 plus 544? Verify with a tool.", "expected": "1000"},
    {"id": 17, "category": "tool_selection", "task": "What is the capital of France?", "expected": "paris"},
    {"id": 18, "category": "tool_selection", "task": "Compute 12 times 12.", "expected": "144"},
    {"id": 19, "category": "tool_selection", "task": "What is the longest river?", "expected": "nile"},
    {"id": 20, "category": "tool_selection", "task": "What is 25 percent of 400?", "expected": "100"},
    {"id": 21, "category": "sequential", "task": "First find the capital of India, then find its population.", "expected": "32"},
    {"id": 22, "category": "sequential", "task": "Calculate 50 times 4, then add 100 to the result.", "expected": "300"},
    {"id": 23, "category": "sequential", "task": "Find the height of Mount Everest, then divide it by 2.", "expected": "4424"},
    {"id": 24, "category": "sequential", "task": "Calculate 10 squared, then multiply the result by 3.", "expected": "300"},
    {"id": 25, "category": "sequential", "task": "Find the length of the longest river, then subtract 650 from it.", "expected": "6000"},
    # ---------------- 25 new tasks ----------------
    {"id": 26, "category": "single_lookup", "task": "What is the capital of Canada?", "expected": "ottawa"},
    {"id": 27, "category": "single_lookup", "task": "Who is the CEO of Apple?", "expected": "cook"},
    {"id": 28, "category": "single_lookup", "task": "What is the largest desert in the world?", "expected": "sahara"},
    {"id": 29, "category": "single_lookup", "task": "What is the population of Sydney?", "expected": "5.3"},
    {"id": 30, "category": "single_lookup", "task": "How many continents are there on Earth?", "expected": "7"},
    {"id": 31, "category": "arithmetic", "task": "What is 640 divided by 16?", "expected": "40"},
    {"id": 32, "category": "arithmetic", "task": "What is 35 percent of 2000?", "expected": "700"},
    {"id": 33, "category": "arithmetic", "task": "What is 18 multiplied by 45?", "expected": "810"},
    {"id": 34, "category": "arithmetic", "task": "What is the square of 31?", "expected": "961"},
    {"id": 35, "category": "arithmetic", "task": "What is 12345 plus 54321?", "expected": "66666"},
    {"id": 36, "category": "multi_step", "task": "What is half the population of Sydney in millions?", "expected": "2.65"},
    {"id": 37, "category": "multi_step", "task": "Find the distance from Earth to the Moon in km and divide it by 1000.", "expected": "384.4"},
    {"id": 38, "category": "multi_step", "task": "What is the population of Cairo plus the population of Sydney, in millions?", "expected": "27.3"},
    {"id": 39, "category": "multi_step", "task": "Find the normal human body temperature in Celsius and multiply it by 10.", "expected": "370"},
    {"id": 40, "category": "multi_step", "task": "Find the number of continents on Earth and multiply it by 25.", "expected": "175"},
    {"id": 41, "category": "tool_selection", "task": "What is 850 minus 350? Verify with a tool.", "expected": "500"},
    {"id": 42, "category": "tool_selection", "task": "What is the capital of Brazil?", "expected": "brasilia"},
    {"id": 43, "category": "tool_selection", "task": "Compute 9 times 111.", "expected": "999"},
    {"id": 44, "category": "tool_selection", "task": "What is the largest ocean?", "expected": "pacific"},
    {"id": 45, "category": "tool_selection", "task": "What is 5 percent of 640?", "expected": "32"},
    {"id": 46, "category": "sequential", "task": "First find the capital of Canada, then find its population.", "expected": "1.4"},
    {"id": 47, "category": "sequential", "task": "Calculate 25 times 8, then divide the result by 4.", "expected": "50"},
    {"id": 48, "category": "sequential", "task": "Find the distance from Earth to the Moon in km, then subtract 4400 from it.", "expected": "380000"},
    {"id": 49, "category": "sequential", "task": "Calculate 15 squared, then add 75 to the result.", "expected": "300"},
    {"id": 50, "category": "sequential", "task": "Find the population of Cairo, then multiply it by 2.", "expected": "44"},
]
print(f"Benchmark A loaded: {len(BENCHMARK_50)} tasks")


# %% ==========================================================================
# CELL 5 — BENCHMARK B: GSM8K (100 problems, calculator tool)
# Standard grade-school math benchmark. The agent must chain calculator
# calls. Scoring: exact numeric match with the gold answer.
# ==============================================================================
from datasets import load_dataset

gsm8k_raw = load_dataset("gsm8k", "main", split="test",
                         trust_remote_code=True)
gsm8k_raw = gsm8k_raw.shuffle(seed=42).select(range(100))

GSM8K_TASKS = []
for i, ex in enumerate(gsm8k_raw):
    gold = float(ex["answer"].split("####")[-1].strip().replace(",", ""))
    GSM8K_TASKS.append({"id": i + 1, "task": ex["question"], "gold": gold})

GSM8K_SYSTEM_PROMPT = """You are a helpful AI agent. You solve math word problems step by step using a calculator tool.

Available tools:
- calculator[expression]: evaluates a math expression, e.g. calculator[3 * (12 + 5)]

Use this exact format:
Thought: <your reasoning>
Action: calculator[<expression>]
Observation: <tool result>
... (repeat Thought/Action/Observation as needed)
Thought: I now know the answer.
Final Answer: <the final number only>

Example:
Task: A shop sells pens at 4 dollars each. Tom buys 3 pens and pays with a 20 dollar bill. How much change does he get?
Thought: First I compute the cost of 3 pens.
Action: calculator[3 * 4]
Observation: 12
Thought: Now I subtract the cost from 20.
Action: calculator[20 - 12]
Observation: 8
Thought: I now know the answer.
Final Answer: 8"""

GSM8K_TOOLS = {"calculator": tool_calculator}

def extract_last_number(text):
    nums = re.findall(r"[-+]?\d[\d,]*\.?\d*", text.replace("$", ""))
    if not nums:
        return None
    try:
        return float(nums[-1].replace(",", "").rstrip("."))
    except ValueError:
        return None

def score_gsm8k(item, answer_text):
    pred = extract_last_number(answer_text)
    return pred is not None and abs(pred - item["gold"]) < 1e-3

print(f"Benchmark B loaded: {len(GSM8K_TASKS)} GSM8K problems")


# %% ==========================================================================
# CELL 6 — BENCHMARK C: HotpotQA distractor (100 questions, search tool)
# Multi-hop QA: the model must chain search calls over 10 provided
# paragraphs (2 gold + 8 distractors) to find the answer.
# Scoring: normalized gold answer contained in normalized prediction
# (inclusion match). Yes/no questions are excluded so the match is clean —
# we state this sampling choice in the paper.
# NOTE: first load downloads ~600 MB; needs cloud notebook Internet ON.
# ==============================================================================
hotpot_raw = load_dataset("hotpot_qa", "distractor", split="validation",
                          trust_remote_code=True)
hotpot_raw = hotpot_raw.shuffle(seed=42)

HOTPOT_TASKS = []
for ex in hotpot_raw:
    ans = ex["answer"].strip()
    if ans.lower() in ("yes", "no") or len(ans) == 0:
        continue
    titles = ex["context"]["title"]
    paragraphs = [" ".join(sents) for sents in ex["context"]["sentences"]]
    HOTPOT_TASKS.append({
        "id": len(HOTPOT_TASKS) + 1,
        "task": ex["question"],
        "gold": ans,
        "titles": titles,
        "paragraphs": paragraphs,
    })
    if len(HOTPOT_TASKS) == 100:
        break

HOTPOT_SYSTEM_PROMPT = """You are a helpful AI agent. You answer questions by searching a set of documents. Some questions need TWO searches: first find one fact, then search again using that fact.

Available tools:
- search[query]: returns the most relevant document passage for the query

Use this exact format:
Thought: <your reasoning>
Action: search[<query>]
Observation: <passage>
... (repeat Thought/Action/Observation as needed)
Thought: I now know the answer.
Final Answer: <a short answer, just the name/date/entity>"""

def make_hotpot_search(item):
    """Per-question search tool over that question's 10 paragraphs."""
    def search(query):
        q_words = set(re.findall(r"\w+", query.lower()))
        scored = []
        for title, para in zip(item["titles"], item["paragraphs"]):
            t_words = set(re.findall(r"\w+", (title + " " + para).lower()))
            title_words = set(re.findall(r"\w+", title.lower()))
            score = len(q_words & t_words) + 3 * len(q_words & title_words)
            scored.append((score, title, para))
        scored.sort(key=lambda x: -x[0])
        best = scored[0]
        if best[0] == 0:
            return "No results found."
        return f"[{best[1]}] {best[2][:600]}"
    return search

def normalize_answer(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def score_hotpot(item, answer_text):
    gold = normalize_answer(item["gold"])
    pred = normalize_answer(answer_text)
    return len(gold) > 0 and gold in pred

print(f"Benchmark C loaded: {len(HOTPOT_TASKS)} HotpotQA questions "
      f"(span answers only)")


# %% ==========================================================================
# CELL 7 — EVALUATION HARNESS (incremental CSV saving every 10 tasks)
# ==============================================================================
RESULTS = {}   # (benchmark, config) -> DataFrame

def evaluate(benchmark, config, items, agent_fn, scorer, save_every=10):
    rows = []
    path = f"/workdir/exp4_{benchmark}_{config}.csv"
    print("=" * 70)
    print(f"RUNNING: {benchmark} | {config} | {len(items)} tasks")
    print("=" * 70)
    for i, item in enumerate(items, 1):
        try:
            ans, steps, calls, lat, trace = agent_fn(item)
        except Exception as e:
            ans, steps, calls, lat = f"AGENT ERROR: {e}", 0, 0, 0.0
        correct = scorer(item, ans)
        rows.append({
            "id": item["id"],
            "category": item.get("category", benchmark),
            "task": item["task"][:120],
            "gold": str(item.get("expected", item.get("gold")))[:60],
            "answer": str(ans)[:120],
            "correct": correct, "steps": steps,
            "tool_calls": calls, "latency_s": round(lat, 2),
        })
        mark = "✅" if correct else "❌"
        print(f"{mark} [{i:03d}/{len(items)}] {lat:5.1f}s  {str(ans)[:55]}")
        if i % save_every == 0 or i == len(items):
            pd.DataFrame(rows).to_csv(path, index=False)
    df = pd.DataFrame(rows)
    RESULTS[(benchmark, config)] = df
    acc = 100 * df["correct"].mean()
    print(f"\n>>> {benchmark} | {config}: {acc:.1f}% "
          f"({df['correct'].sum()}/{len(df)}), "
          f"avg latency {df['latency_s'].mean():.2f}s")
    print(f">>> saved to {path}\n")
    return df

# agent wrappers per benchmark
def agent_custom(item):
    return run_agent(item["task"], CUSTOM_TOOLS, CUSTOM_SYSTEM_PROMPT,
                     max_steps=6, max_new_tokens=200)

def agent_gsm8k(item):
    return run_agent(item["task"], GSM8K_TOOLS, GSM8K_SYSTEM_PROMPT,
                     max_steps=8, max_new_tokens=260)

def agent_hotpot(item):
    tools = {"search": make_hotpot_search(item)}
    return run_agent(item["task"], tools, HOTPOT_SYSTEM_PROMPT,
                     max_steps=6, max_new_tokens=220)

def score_custom(item, ans):
    return item["expected"].lower() in ans.lower()

print("Harness ready.")


# %% ==========================================================================
# CELL 8 — EVAL 1/6: INT4 base on Custom-50   (~10-20 min)
# ==============================================================================
evaluate("custom50", "int4", BENCHMARK_50, agent_custom, score_custom)


# %% ==========================================================================
# CELL 9 — EVAL 2/6: INT4 base on GSM8K   (~40-70 min)
# ==============================================================================
evaluate("gsm8k", "int4", GSM8K_TASKS, agent_gsm8k, score_gsm8k)


# %% ==========================================================================
# CELL 10 — EVAL 3/6: INT4 base on HotpotQA   (~30-50 min)
# ==============================================================================
evaluate("hotpotqa", "int4", HOTPOT_TASKS, agent_hotpot, score_hotpot)


# %% ==========================================================================
# CELL 11 — TRAIN QLoRA (same 15 held-out examples as Experiment 3b, ~2 min)
# CRITICAL: we do NOT train on GSM8K or HotpotQA data. The claim we are
# testing: "15 generic ReAct demos generalize to standard benchmarks."
# ==============================================================================
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

def make_example(task, trace):
    return {"text": (f"<|system|>\n{CUSTOM_SYSTEM_PROMPT}<|end|>\n"
                     f"<|user|>\nTask: {task}<|end|>\n"
                     f"<|assistant|>\n{trace}<|end|>")}

TRAIN_EXAMPLES = [
    make_example("What is the population of London?",
        "Thought: I need to look up the population of London.\n"
        "Action: search[population of London]\n"
        "Observation: The population of London is 9 million.\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The population of London is 9 million."),
    make_example("What is the capital of Germany?",
        "Thought: I need to find the capital of Germany.\n"
        "Action: search[capital of Germany]\n"
        "Observation: The capital of Germany is Berlin.\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The capital of Germany is Berlin."),
    make_example("What is 125 multiplied by 8?",
        "Thought: This is a math problem, I should use the calculator.\n"
        "Action: calculator[125 * 8]\n"
        "Observation: 1000\n"
        "Thought: I now know the answer.\n"
        "Final Answer: 125 multiplied by 8 is 1000."),
    make_example("What is 20 percent of 450?",
        "Thought: I need to compute 20% of 450 using the calculator.\n"
        "Action: calculator[0.20 * 450]\n"
        "Observation: 90.0\n"
        "Thought: I now know the answer.\n"
        "Final Answer: 20 percent of 450 is 90."),
    make_example("What is triple the population of London in millions?",
        "Thought: First I need the population of London, then I multiply it by 3.\n"
        "Action: search[population of London]\n"
        "Observation: The population of London is 9 million.\n"
        "Thought: Now I multiply 9 by 3 using the calculator.\n"
        "Action: calculator[9 * 3]\n"
        "Observation: 27\n"
        "Thought: I now know the answer.\n"
        "Final Answer: Triple the population of London is 27 million."),
    make_example("Find the population of Mumbai and subtract 3 million from it.",
        "Thought: First I search for Mumbai's population, then subtract 3 million.\n"
        "Action: search[population of Mumbai]\n"
        "Observation: The population of Mumbai is 21 million.\n"
        "Thought: Now I subtract 3 from 21 using the calculator.\n"
        "Action: calculator[21 - 3]\n"
        "Observation: 18\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The population of Mumbai minus 3 million is 18 million."),
    make_example("How many moons does Jupiter have, divided by 5?",
        "Thought: First I need to find how many moons Jupiter has.\n"
        "Action: search[number of moons of Jupiter]\n"
        "Observation: Jupiter has 95 moons.\n"
        "Thought: Now I divide 95 by 5.\n"
        "Action: calculator[95 / 5]\n"
        "Observation: 19.0\n"
        "Thought: I now know the answer.\n"
        "Final Answer: Jupiter's 95 moons divided by 5 is 19."),
    make_example("What is the population of London plus the population of Mumbai, in millions?",
        "Thought: I need both populations. First, London.\n"
        "Action: search[population of London]\n"
        "Observation: The population of London is 9 million.\n"
        "Thought: Next, Mumbai.\n"
        "Action: search[population of Mumbai]\n"
        "Observation: The population of Mumbai is 21 million.\n"
        "Thought: Now I add 9 and 21.\n"
        "Action: calculator[9 + 21]\n"
        "Observation: 30\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The combined population of London and Mumbai is 30 million."),
    make_example("Search for the depth of the Mariana Trench in meters and divide it by 100.",
        "Thought: First I search for the depth of the Mariana Trench.\n"
        "Action: search[depth of Mariana Trench in meters]\n"
        "Observation: The Mariana Trench is 10935 meters deep.\n"
        "Thought: Now I divide 10935 by 100.\n"
        "Action: calculator[10935 / 100]\n"
        "Observation: 109.35\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The depth of the Mariana Trench divided by 100 is 109.35."),
    make_example("Find the freezing point of water in Fahrenheit and multiply it by 4.",
        "Thought: First I search for the freezing point of water in Fahrenheit.\n"
        "Action: search[freezing point of water in Fahrenheit]\n"
        "Observation: The freezing point of water is 32 degrees Fahrenheit.\n"
        "Thought: Now I multiply 32 by 4.\n"
        "Action: calculator[32 * 4]\n"
        "Observation: 128\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The freezing point of water in Fahrenheit multiplied by 4 is 128."),
    make_example("What is 777 plus 223? Do not guess, verify it.",
        "Thought: This is arithmetic, so the correct tool is the calculator, not search.\n"
        "Action: calculator[777 + 223]\n"
        "Observation: 1000\n"
        "Thought: I now know the answer.\n"
        "Final Answer: 777 plus 223 is 1000."),
    make_example("Who is the current CEO of Google?",
        "Thought: This is a factual lookup, so the correct tool is search, not calculator.\n"
        "Action: search[current CEO of Google]\n"
        "Observation: The CEO of Google is Sundar Pichai.\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The current CEO of Google is Sundar Pichai."),
    make_example("Compute the cube of 6.",
        "Thought: Cubing a number is math, so I use the calculator.\n"
        "Action: calculator[6 ** 3]\n"
        "Observation: 216\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The cube of 6 is 216."),
    make_example("First find the capital of Australia, then find its population.",
        "Thought: Step one: find the capital of Australia.\n"
        "Action: search[capital of Australia]\n"
        "Observation: The capital of Australia is Canberra.\n"
        "Thought: Step two: find the population of Canberra.\n"
        "Action: search[population of Canberra]\n"
        "Observation: The population of Canberra is 0.45 million.\n"
        "Thought: I now know the answer.\n"
        "Final Answer: The capital of Australia is Canberra and its population is 0.45 million."),
    make_example("Calculate 30 times 6, then subtract 80 from the result.",
        "Thought: Step one: compute 30 times 6.\n"
        "Action: calculator[30 * 6]\n"
        "Observation: 180\n"
        "Thought: Step two: subtract 80 from 180.\n"
        "Action: calculator[180 - 80]\n"
        "Observation: 100\n"
        "Thought: I now know the answer.\n"
        "Final Answer: 30 times 6 minus 80 is 100."),
]

train_dataset = Dataset.from_list(TRAIN_EXAMPLES)
print(f"Training examples: {len(train_dataset)}")

model.config.use_cache = False
tokenizer.padding_side = "right"
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules=["qkv_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir="/workdir/exp4-qlora",
    num_train_epochs=8,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    seed=42,
    report_to="none",          # set to "wandb" if you ran wandb.login()
)

trainer = SFTTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    dataset_text_field="text", max_seq_length=768, packing=False,
    tokenizer=tokenizer,
)
trainer.train()
trainer.model.save_pretrained("/workdir/exp4-lora-adapters")

# switch back to inference mode
model.config.use_cache = True
model.eval()
tokenizer.padding_side = "left"
gc.collect(); torch.cuda.empty_cache()
print("QLoRA training done — model is now INT4 + QLoRA for the next cells.")


# %% ==========================================================================
# CELL 12 — EVAL 4/6: INT4+QLoRA on Custom-50
# ==============================================================================
evaluate("custom50", "int4_qlora", BENCHMARK_50, agent_custom, score_custom)


# %% ==========================================================================
# CELL 13 — EVAL 5/6: INT4+QLoRA on GSM8K
# ==============================================================================
evaluate("gsm8k", "int4_qlora", GSM8K_TASKS, agent_gsm8k, score_gsm8k)


# %% ==========================================================================
# CELL 14 — EVAL 6/6: INT4+QLoRA on HotpotQA
# ==============================================================================
evaluate("hotpotqa", "int4_qlora", HOTPOT_TASKS, agent_hotpot, score_hotpot)


# %% ==========================================================================
# CELL 15 — TABLES 4 AND 5
# ==============================================================================
def load_result(benchmark, config):
    key = (benchmark, config)
    if key in RESULTS:
        return RESULTS[key]
    path = f"/workdir/exp4_{benchmark}_{config}.csv"
    return pd.read_csv(path)   # fallback if the session was restarted

# ---------- TABLE 4: Custom-50 by category ----------
rows = []
for config, label in [("int4", "INT4"), ("int4_qlora", "INT4 + QLoRA")]:
    df = load_result("custom50", config)
    row = {"Config": label,
           "Overall %": round(100 * df["correct"].mean(), 1),
           "Avg Latency (s)": round(df["latency_s"].mean(), 2)}
    for cat in ["single_lookup", "arithmetic", "multi_step",
                "tool_selection", "sequential"]:
        sub = df[df["category"] == cat]
        row[cat + " %"] = round(100 * sub["correct"].mean(), 1)
    rows.append(row)
table4 = pd.DataFrame(rows)
print("\nTABLE 4 — Custom 50-task agentic benchmark (held-out)")
print(table4.to_string(index=False))
table4.to_csv("/workdir/table4_custom50.csv", index=False)

# ---------- TABLE 5: Standard benchmarks ----------
rows = []
for bench, label in [("gsm8k", "GSM8K (100)"), ("hotpotqa", "HotpotQA (100)")]:
    for config, clabel in [("int4", "INT4"), ("int4_qlora", "INT4 + QLoRA")]:
        df = load_result(bench, config)
        rows.append({
            "Benchmark": label, "Config": clabel,
            "Accuracy %": round(100 * df["correct"].mean(), 1),
            "Avg Latency (s)": round(df["latency_s"].mean(), 2),
            "Avg Tool Calls": round(df["tool_calls"].mean(), 2),
        })
table5 = pd.DataFrame(rows)
print("\nTABLE 5 — Standard benchmarks (agent not fine-tuned on either)")
print(table5.to_string(index=False))
table5.to_csv("/workdir/table5_standard_benchmarks.csv", index=False)

# ---------- optional W&B logging ----------
try:
    import wandb
    run = wandb.init(project="slm-agentic-ai", name="exp4-external-validation")
    run.log({"table4_custom50": wandb.Table(dataframe=table4),
             "table5_standard": wandb.Table(dataframe=table5)})
    run.finish()
    print("\nLogged to W&B ✅")
except Exception as e:
    print(f"\nW&B logging skipped: {e}")


# %% ==========================================================================
# CELL 16 — ZIP EVERYTHING FOR DOWNLOAD
# ==============================================================================
"""
!zip -r /workdir/exp4_all_results.zip /workdir/exp4_*.csv /workdir/table4_custom50.csv /workdir/table5_standard_benchmarks.csv /workdir/exp4-lora-adapters
!ls -lh /workdir/
"""

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded. Footprint: 2.21 GB
Agent engine ready.
Benchmark A loaded: 50 tasks


Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

...

Step,Training Loss
1,1.631000
2,1.564700
3,1.393000
4,1.140700
5,1.087000
6,0.890400
7,0.816000
8,0.658800
9,0.604000
10,0.497400


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


QLoRA training done — model is now INT4 + QLoRA for the next cells.
RUNNING: custom50 | int4_qlora | 50 tasks
✅ [001/50]   5.4s  The population of Paris is 2.1 million.
✅ [002/50]   4.7s  The capital of Japan is Tokyo.
✅ [003/50]   5.1s  Satya Nadella.
✅ [004/50]   5.5s  The tallest mountain is Mount Everest at 8849 meters.
✅ [005/50]   5.7s  The boiling point of water is 100 degrees Celsius.
✅ [006/50]   7.0s  340 multiplied by 25 is 8500.
✅ [007/50]  10.0s  15 percent of 8000 is 1200.
✅ [008/50]   6.7s  999 plus 111 is 1110.
✅ [009/50]   5.5s  The square of 47 is 2209.
✅ [010/50]   6.6s  7200 divided by 8 is 900.
✅ [011/50]   5.1s  Double the population of Paris is 4.2 million.
✅ [012/50]   5.3s  The population of Tokyo plus 5 million is 19 million.
❌ [013/50]  20.9s  Observation: 7600000000
✅ [014/50]   8.4s  The population of Paris plus the population of Tokyo is
✅ [015/50]  11.1s  The speed of light divided by 1000 is 299.792 km/s.
✅ [016/50]   6.7s  456 plus 544 equals 1000.
✅ [0


Logged to W&B ✅


'\n!zip -r /workdir/exp4_all_results.zip /workdir/exp4_*.csv /workdir/table4_custom50.csv /workdir/table5_standard_benchmarks.csv /workdir/exp4-lora-adapters\n!ls -lh /workdir/\n'